<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/AN%C3%81LISE_DE_ESTRUTURA_SECUND%C3%81RIA_(DSSP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# ANÁLISE DE ESTRUTURA SECUNDÁRIA (DSSP) AUTOMATIZADA
# Suporta múltiplos arquivos de topologia/trajectória (triplicatas Apo e AG73)
# ================================================================

# 1. Instalar dependências
!pip install -q MDAnalysis biopython matplotlib seaborn pandas

import MDAnalysis as mda
from MDAnalysis.analysis import dssp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import os

WORKDIR = "/content/dssp_analysis"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)


In [ ]:
# 2. Upload de múltiplos arquivos (.pdb e trajetórias .xtc/.trr)
print("📁 Faça upload dos arquivos de topologia (.pdb) e trajetória (.xtc/.trr) das triplicatas (Apo e AG73)")
uploaded = files.upload()
uploaded_files = list(uploaded.keys())

# Identificar topologia e trajetórias
top_files = [f for f in uploaded_files if f.endswith(('.pdb', '.gro'))]
traj_files = [f for f in uploaded_files if f.endswith(('.xtc', '.trr', '.dcd'))]

if len(top_files) != len(traj_files):
    print("⚠️ Atenção: O número de topologias e trajetórias não coincide. Verifique os arquivos.")

In [ ]:
# 3. Separar sistemas por tipo (Apo ou AG73)
systems = {"Apo": [], "AG73": []}
for top, traj in zip(top_files, traj_files):
    if "Apo" in top:
        systems["Apo"].append((top, traj))
    else:
        systems["AG73"].append((top, traj))
print(f"Sistemas detectados: { {k: len(v) for k,v in systems.items()} }")

# 4. Função para cálculo de DSSP
def calc_dssp(top_file, traj_file, ligand_resname="AG73"):
    u = mda.Universe(top_file, traj_file)
    ligand = u.select_atoms(f"resname {ligand_resname}")
    dssp_analysis = dssp.DSSP(u, select=f"resname {ligand_resname}")
    dssp_analysis.run()
    dssp_data = np.array(dssp_analysis.results.secondary_structures)
    mapping = {'H':2, 'G':2, 'I':2, 'E':1, 'B':1, 'T':0, 'S':0, '-':0, 'C':0}
    numeric_dssp = np.vectorize(lambda x: mapping.get(x, 0))(dssp_data)
    residue_ids = [res.resid for res in ligand.residues]
    time_ns = np.arange(dssp_data.shape[0]) * (u.trajectory.dt / 1000)
    df_dssp = pd.DataFrame(numeric_dssp.T, index=residue_ids, columns=[f"{t:.2f} ns" for t in time_ns])
    return df_dssp

# 5. Calcular DSSP para todos os sistemas e salvar CSVs
output_dir = os.path.join(WORKDIR, "DSSP_matrices")
os.makedirs(output_dir, exist_ok=True)

results_dssp = {}
for sys_name, files_list in systems.items():
    for i, (top, traj) in enumerate(files_list, start=1):
        df = calc_dssp(top, traj)
        fname = f"{sys_name}_rep{i}_DSSP.csv"
        df.to_csv(os.path.join(output_dir, fname))
        results_dssp[fname] = df
        print(f"✅ {fname} salvo ({df.shape[0]} resíduos × {df.shape[1]} frames)")

# 6. Fazer download de todos os CSVs
for fname in results_dssp.keys():
    files.download(os.path.join(output_dir, fname))




In [ ]:
# 7. Plotagem automática dos heatmaps individuais
sns.set(style="white")
cmap = sns.color_palette(["white", "red"])

for fname, df in results_dssp.items():
    plt.figure(figsize=(14, 6))
    sns.heatmap(df, cmap=cmap, cbar_kws={'label':'Estrutura Secundária (0=Coil,1=Folha,2=Hélice)'})
    plt.xlabel("Tempo (ns)")
    plt.ylabel("Número do Resíduo")
    plt.title(fname.replace(".csv",""))
    plt.tight_layout()
    png_file = fname.replace(".csv", "_heatmap.png")
    plt.savefig(png_file, dpi=600)
    plt.show()
    files.download(png_file)

# 8. Heatmap combinado de todas as triplicatas (opcional)
fig, axes = plt.subplots(len(results_dssp),1,figsize=(14,4*len(results_dssp)), sharex=True)
if len(results_dssp)==1:
    axes = [axes]

for ax, (fname, df) in zip(axes, results_dssp.items()):
    sns.heatmap(df, ax=ax, cmap=cmap, cbar=False, xticklabels=100, yticklabels=10)
    ax.set_title(fname.replace(".csv",""), fontsize=12, fontweight="bold")
    ax.set_ylabel("Residuo")
axes[-1].set_xlabel("Tempo (ns)")
plt.tight_layout()
combined_file = "Combined_DSSP_heatmap.png"
plt.savefig(combined_file, dpi=600)
plt.show()
files.download(combined_file)

print("✅ Análise DSSP completa: heatmaps individuais e combinados gerados.")